---
title: Глава 4. Фильтрация
subtitle: SQL Lab in JupyterLab
# license: CC-BY-4.0
github: https://github.com/magus1968/learning-sql
subject: Technical Portfolio
# subject: SQL Learning & Tooling
venue: GitHub & GitVerse Pages
abstract: |
  В терминале после вывода последней строки данных *mysql* выводит информацию о количестве возвращенных строк и затраченном времени. Например, `5462 rows in set (0.01 sec)`. Количество строк мы уже отображали в предыдущих главах `5462 rows affected.` В этой главе добавим отображение времени.
authors:
  - name: Alex Smirnov
    email: a@smirnovs.pro
    corresponding: true
    affiliations: Data & BI Analyst
      # - Data Analyst
      # - BI Analyst
      # - Business Analyst
      # - Independent Researcher
date: 2026-08-01
abbreviations:
    MyST: Markedly Structured Text
    Jupyter Book: Build static Web-books
    JupySQL: Run & highlight SQL in Jupyter
---

In [1]:
from sqlalchemy import create_engine
from sqlalchemy.engine import URL


connection_url = URL.create(
    drivername="mysql+pymysql",
    host="localhost",
    port=3306,
    database="sakila",
    username="root",
    password="*UHB5rdx",
)

engine = create_engine(connection_url)

%load_ext sql

%config SqlMagic.displaylimit = 20
%config SqlMagic.displaycon = False
# %config SqlMagic.feedback = False

%sql engine

print("SQLAlchemy - подключение создано")
print("JupySQL - успешно подключен через SQLAlchemy Engine!")

SQLAlchemy - подключение создано
JupySQL - успешно подключен через SQLAlchemy Engine!


## Вычисление условий

```sql
WHERE first_name = 'STEVEN' AND create_date > '2006-01-01`

WHERE first_name = 'STEVEN' OR create_date > '2006-01-01`

-- Использование скобок
WHERE (first_name = 'STEVEN' OR last_name = 'YOUNG')
  AND create_date > '2006-01-01')

-- Использование оператора `not`
WHERE NOT (first_name = 'STEVEN' OR last_name = 'YOUNG')
  AND create_date > '2006-01-01')

WHERE first_name <> 'STEVEN' AND last_name <> 'YOUNG'    -- OR заменен на AND !
  AND create_date > '2006-01-01')
```

## Типы условий

### Условия равенства

```sql
title = 'RIVER OUTLAW'
fed_id = '111-11-1111'
amount = 375.25
film_id = (SELECT film_id FROM film WHERE title = 'RIVER OUTLAW')
```

In [10]:
%%time
%%sql
SELECT c.email
FROM customer c
  INNER JOIN rental r
  ON c.customer_id = r.customer_id
WHERE date(r.rental_date) = '2005-06-14';

16 rows affected.

CPU times: total: 0 ns
Wall time: 45 ms


email
CATHERINE.CAMPBELL@sakilacustomer.org
JOYCE.EDWARDS@sakilacustomer.org
AMBER.DIXON@sakilacustomer.org
JEANETTE.GREENE@sakilacustomer.org
MINNIE.ROMERO@sakilacustomer.org
GWENDOLYN.MAY@sakilacustomer.org
SONIA.GREGORY@sakilacustomer.org
MIRIAM.MCKINNEY@sakilacustomer.org
CHARLES.KOWALSKI@sakilacustomer.org
DANIEL.CABRAL@sakilacustomer.org


### Условия неравенства

In [6]:
import polars as pl
print(f"Polars ver. {pl.__version__}")

Polars ver. 1.43.1


In [43]:
%config SqlMagic.autopolars = True

In [13]:
%%sql
SELECT c.email
FROM customer c
  INNER JOIN rental r
  ON c.customer_id = r.customer_id
WHERE date(r.rental_date) <> '2005-06-14';

16028 rows affected.

email
str
"""MARY.SMITH@sakilacustomer.org"""
"""MARY.SMITH@sakilacustomer.org"""
"""MARY.SMITH@sakilacustomer.org"""
"""MARY.SMITH@sakilacustomer.org"""
"""MARY.SMITH@sakilacustomer.org"""
…
"""AUSTIN.CINTRON@sakilacustomer.…"
"""AUSTIN.CINTRON@sakilacustomer.…"
"""AUSTIN.CINTRON@sakilacustomer.…"


```sql
-- Модификация условий равенства/неравенства

DELETE FROM rental
WHERE year(rental_date) = 2024;

DELETE FROM rental
WHERE year(rental_date) <> 2005 AND year(rental_date) <> 2006;

### Условие диапазона

In [15]:
%%sql
SELECT customer_id, rental_id
FROM rental
WHERE rental_date < '2005-05-25';

8 rows affected.

customer_id,rental_id
130,1
459,2
408,3
333,4
222,5
549,6
269,7
239,8


In [17]:
%%sql
SELECT customer_id, rental_date
FROM rental
WHERE rental_date <= '2005-06-16'
  AND rental_date >= '2005-06-14';

364 rows affected.

customer_id,rental_date
i64,datetime[μs]
416,2005-06-14 22:53:33
516,2005-06-14 22:55:13
239,2005-06-14 23:00:34
285,2005-06-14 23:07:08
310,2005-06-14 23:09:38
…,…
148,2005-06-15 23:20:26
237,2005-06-15 23:36:37
155,2005-06-15 23:55:27


#### Оператор _beetween_

In [18]:
%%sql
SELECT customer_id, rental_date
FROM rental
WHERE rental_date BETWEEN '2005-06-14' AND '2005-06-16';

364 rows affected.

customer_id,rental_date
i64,datetime[μs]
416,2005-06-14 22:53:33
516,2005-06-14 22:55:13
239,2005-06-14 23:00:34
285,2005-06-14 23:07:08
310,2005-06-14 23:09:38
…,…
148,2005-06-15 23:20:26
237,2005-06-15 23:36:37
155,2005-06-15 23:55:27


In [19]:
%%sql
SELECT customer_id, rental_date
FROM rental
WHERE rental_date BETWEEN '2005-06-16' AND '2005-06-14';

customer_id,rental_date
null,null


In [20]:
%%sql
SELECT customer_id, rental_date
FROM rental
WHERE rental_date >= '2005-06-16'
  AND rental_date <= '2005-06-14';

customer_id,rental_date
null,null


In [21]:
%%sql
SELECT customer_id, payment_id, amount
FROM payment
WHERE amount BETWEEN 10.0 AND 11.99;

114 rows affected.

customer_id,payment_id,amount
i64,i64,"decimal[38,2]"
2,44,10.99
3,69,10.99
12,324,10.99
13,342,11.99
21,551,10.99
…,…,…
572,15313,10.99
573,15353,10.99
591,15821,11.99


#### Строковые диапазоны

In [23]:
%%sql
SELECT last_name, first_name
FROM customer
WHERE last_name BETWEEN 'FA' AND 'FR';

18 rows affected.

last_name,first_name
FARNSWORTH,JOHN
FENNELL,ALEXANDER
FERGUSON,BERTHA
FERNANDEZ,MELINDA
FIELDS,VICKI
FISHER,CINDY
FLEMING,MYRTLE
FLETCHER,MAE
FLORES,JULIA
FORD,CRYSTAL


In [25]:
%%sql
SELECT last_name, first_name
FROM customer
WHERE last_name BETWEEN 'FA' AND 'FRB';

22 rows affected.

last_name,first_name
str,str
"""FARNSWORTH""","""JOHN"""
"""FENNELL""","""ALEXANDER"""
"""FERGUSON""","""BERTHA"""
"""FERNANDEZ""","""MELINDA"""
"""FIELDS""","""VICKI"""
…,…
"""FOX""","""HOLLY"""
"""FRALEY""","""JUAN"""
"""FRANCISCO""","""JOEL"""


### Условия членства

In [26]:
%%sql
SELECT title, rating
FROM film
WHERE rating = 'G' OR rating = 'PG';

372 rows affected.

title,rating
str,str
"""ACADEMY DINOSAUR""","""PG"""
"""ACE GOLDFINGER""","""G"""
"""AFFAIR PREJUDICE""","""G"""
"""AFRICAN EGG""","""G"""
"""AGENT TRUMAN""","""PG"""
…,…
"""WON DARES""","""PG"""
"""WONDERLAND CHRISTMAS""","""PG"""
"""WORDS HUNTER""","""PG"""


#### Оператор _in_

In [28]:
%%sql
SELECT title, rating
FROM film
WHERE rating IN ('G', 'PG');

372 rows affected.

title,rating
str,str
"""ACADEMY DINOSAUR""","""PG"""
"""ACE GOLDFINGER""","""G"""
"""AFFAIR PREJUDICE""","""G"""
"""AFRICAN EGG""","""G"""
"""AGENT TRUMAN""","""PG"""
…,…
"""WON DARES""","""PG"""
"""WONDERLAND CHRISTMAS""","""PG"""
"""WORDS HUNTER""","""PG"""


#### Использование подзапросов

In [30]:
%%sql
SELECT title, rating
FROM film
WHERE rating IN (SELECT rating FROM film WHERE title LIKE '%PET%');

372 rows affected.

title,rating
str,str
"""ACADEMY DINOSAUR""","""PG"""
"""ACE GOLDFINGER""","""G"""
"""AFFAIR PREJUDICE""","""G"""
"""AFRICAN EGG""","""G"""
"""AGENT TRUMAN""","""PG"""
…,…
"""WON DARES""","""PG"""
"""WONDERLAND CHRISTMAS""","""PG"""
"""WORDS HUNTER""","""PG"""


```sql
-- Пояснение подзапроса
SELECT title, rating FROM film WHERE title LIKE '%PET%';

-- | title           | rating |
-- | --------------- | ------ |
-- | "MALKOVICH PET" | "G"    |
-- | "MUPPET MILE"   | "PG"   |
-- | "PET HAUNTING"  | "PG"   |
```

#### Использование _not in_

In [ ]:
%%sql
SELECT title, rating
FROM film
WHERE rating NOT IN ('PG-13', 'R', 'NC-17');

### Условия соответствия

In [38]:
%%sql
SELECT last_name, first_name
FROM customer
WHERE left(last_name, 1) = 'Q';

3 rows affected.

last_name,first_name
QUALLS,STEPHEN
QUINTANILLA,ROGER
QUIGLEY,TROY


#### Использование подстановочных знаков

```text
| Подстановочный символ | Соответствие                         |
| --------------------- | ------------------------------------ |
|  _                    | В точности один символ               |
|  %                    | Любое количество символов, включая 0 |


# Примеры выражений поиска

| Выражение поиска | Интерпретация                                         |
| ---------------- | ----------------------------------------------------- |
| F%               | Строка, начинающаяся с F                              |
| %f               | Строка, заканчивающаяся f                             |
| %bas%            | Строка, содержащая подстроку `bas`                    |
| __t_             | Строка из 4 символов с t в третьей позиции            |
| ___-__-____      | Строка из 11 символов с дефисами в 4-й и 7-й позициях |
```

In [39]:
%%sql
SELECT last_name, first_name
FROM customer
WHERE last_name LIKE '_A_T%S';

3 rows affected.

last_name,first_name
MATTHEWS,ERICA
WALTERS,CASSANDRA
WATTS,SHELLY


In [41]:
%%sql
SELECT last_name, first_name
FROM customer
WHERE last_name LIKE 'Q%' OR last_name LIKE 'Y%';

6 rows affected.

last_name,first_name
QUALLS,STEPHEN
QUIGLEY,TROY
QUINTANILLA,ROGER
YANEZ,LUIS
YEE,MARVIN
YOUNG,CYNTHIA


#### Использование регулярных выражений

In [42]:
%%sql
SELECT last_name, first_name
FROM customer
WHERE last_name REGEXP '^[QY]';

6 rows affected.

last_name,first_name
YOUNG,CYNTHIA
QUALLS,STEPHEN
QUINTANILLA,ROGER
YANEZ,LUIS
YEE,MARVIN
QUIGLEY,TROY


### Этот таинственный _null_

In [46]:
%%sql
SELECT rental_id, customer_id
FROM rental
WHERE return_date IS NULL;

183 rows affected.

rental_id,customer_id
i64,i64
11496,155
11541,335
11563,83
11577,219
11593,99
…,…
15862,215
15867,505
15875,41


:::{div}
:class: text-center border rounded

Выражение может _быть_ null, но оно никогда не может быть _равным_ null:
:::

In [47]:
%%sql
SELECT rental_id, customer_id
FROM rental
WHERE return_date = NULL;

rental_id,customer_id
null,null


In [48]:
%%sql
SELECT rental_id, customer_id, return_date
FROM rental
WHERE return_date IS NOT NULL;

15861 rows affected.

rental_id,customer_id,return_date
i64,i64,datetime[μs]
1,130,2005-05-26 22:04:30
2,459,2005-05-28 19:40:33
3,408,2005-06-01 22:12:39
4,333,2005-06-03 01:43:41
5,222,2005-06-02 04:33:21
…,…,…
16045,14,2005-08-25 23:54:26
16046,74,2005-08-27 18:02:47
16047,114,2005-08-25 02:48:48


In [53]:
%%sql
SELECT rental_id, customer_id, return_date
FROM rental
WHERE return_date NOT BETWEEN '2005-05-01' AND '2005-09-01';

62 rows affected.

rental_id,customer_id,return_date
i64,i64,datetime[μs]
15365,327,2005-09-01 03:14:17
15388,50,2005-09-01 03:50:23
15392,410,2005-09-01 01:14:15
15401,103,2005-09-01 03:44:10
15415,204,2005-09-01 02:05:56
…,…,…
16005,466,2005-09-02 02:35:22
16020,311,2005-09-01 18:17:33
16033,226,2005-09-01 02:36:15


In [ ]:
%%sql
SELECT rental_id, customer_id, return_date
FROM rental
WHERE return_date IS NULL
  OR return_date NOT BETWEEN '2005-05-01' AND '2005-09-01';

:::{div}
:class: text-xs
Вывод ошибки _`ComputeError`_ убрал _**Clear Cell Output**_ и добавил заметку:
:::

:::{error} Библиотека Polars упала с ошибкой `ComputeError`
:class: dropdown
:open: true
Polars не смог автоматически распознать тип данных для столбца _return_date_, так как функция парсинга Polars _(iterable_to_pydf)_ споткнулась: она начала строить колонку на основе первых попавшихся `None`, зафиксировала схему, а затем внезапно встретила реальный объект даты `datetime.datetime`. Произошел **конфликт типов прямо во время сборки таблицы**, из-за чего библиотека и выбросила `ComputeError`.

- Можем просто отключить автоконвертацию в `Polars DataFrame` и получать стандартный вывод JupySQL без усечения через многоточие большого количества строк, закончив эксперименты с Polars;
- Я все-таки хочу сохранить усечение в большом выводе, поэтому вернусь к старому доброму `Pandas DataFrame`, который менее строго типизирован. Пропуски в датах Pandas приведет к специальному маркеру `NaT` _(Not a Time)_ – стандартному обозначению пустого временного значения для Pandas.
:::

In [66]:
%config SqlMagic.autopolars = False

In [67]:
import pandas as pd
print(f"Pandas ver. {pd.__version__}")

Pandas ver. 3.0.3


In [69]:
%config SqlMagic.autopandas = True

In [70]:
%%sql
SELECT rental_id, customer_id, return_date
FROM rental
WHERE return_date IS NULL
  OR return_date NOT BETWEEN '2005-05-01' AND '2005-09-01';

245 rows affected.

,rental_id,customer_id,return_date
0,11496,155,NaT
1,11541,335,NaT
2,11563,83,NaT
3,11577,219,NaT
4,11593,99,NaT
...,...,...,...
240,16005,466,2005-09-02 02:35:22
241,16020,311,2005-09-01 18:17:33
242,16033,226,2005-09-01 02:36:15
243,16037,45,2005-09-01 02:48:04


## Упражнения

Предлагаемые упражнения призваны закрепить понимание условий фильтрации. В первых двух упражнениях потребуется подмножество строк из таблицы _payment_:

```text
| payment_id | customer_id | amount | date(payment date) |
| ---------- | ----------- | ------ | ------------------ |
| 101        | 4           | 8.99   | 2005-08-19         |
| 102        | 4           | 1.99   | 2005-08-19         |
| 103        | 4           | 2.99   | 2005-08-20         |
| 104        | 4           | 6.99   | 2005-08-20         |
| 105        | 4           | 4.99   | 2005-08-21         |
| 106        | 4           | 2.99   | 2005-08-22         |
| 107        | 4           | 1.99   | 2005-08-23         |
| 108        | 5           | 0.99   | 2005-05-29         |
| 109        | 5           | 6.99   | 2005-05-31         |
| 110        | 5           | 1.99   | 2005-05-31         |
| 111        | 5           | 3.99   | 2005-06-15         |
| 112        | 5           | 2.99   | 2005-06-16         |
| 113        | 5           | 4.99   | 2005-06-17         |
| 114        | 5           | 2.99   | 2005-06-19         |
| 115        | 5           | 4.99   | 2005-06-20         |
| 116        | 5           | 4.99   | 2005-07-06         |
| 117        | 5           | 2.99   | 2005-07-08         |
| 118        | 5           | 4.99   | 2005-07-09         |
| 119        | 5           | 5.99   | 2005-07-09         |
| 120        | 5           | 1.99   | 2005-07-09         |
```

### Упражнение 4.1
Какие из идентификаторов платежей будут возвращены при следующих условиях фильтрации?
```sql
customer_id <> 5
  AND (amount > 8 OR date(payment_date) = '2005-08-23')
```

```text
| payment_id | customer_id | amount | date(payment date) |
| ---------- | ----------- | ------ | ------------------ |
| 101        | 4           | 8.99   | 2005-08-19         |
| 107        | 4           | 1.99   | 2005-08-23         |
```

---

### Упражнение 4.2
Какие из идентификаторов платежей будут возвращены при следующих условиях фильтрации?
```text
customer_id = 5 AND
  NOT (amount > 6 OR date(payment_date) = '2005-06-19')
```

In [74]:
# Решение



---

### Упражнение 4.3
Создайте запрос, который извлекает из таблицы _payments_ все строки, в которых сумма равна 1.98, 7.98 или 9.98.

In [73]:
# Решение



---

### Упражнение 4.4
Создайте запрос, который находит всех клиентов, в фамилиях которых содержится буква `А` во второй позиции и буква `W` – в любом месте после `А`.

In [72]:
# Решение



---